In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import seaborn as sns
from scipy import signal

class DroneDataLoader:
    def __init__(self, data_path='..'):
        self.data_path = data_path
        self.index_path = os.path.join(data_path, 'data/dataset_index.csv')
        if not os.path.exists(self.index_path):
            print(f"Index file not found at {self.index_path}")
            return
        
        self.df = pd.read_csv(self.index_path)
        print(f"Registry successfully loaded. Available records: {len(self.df)}")

    def get_samples(self, category=None, freq=None):
        filtered = self.df
        if category:
            filtered = filtered[filtered['category'] == category]
        if freq:
            filtered = filtered[filtered['freq_ghz'] == freq]
        return filtered
    
    def load_data_pair(self, row):
        vtx_path = os.path.join(self.data_path, row['csv_vtx_path'].lstrip('./')) if pd.notna(row['csv_vtx_path']) else None
        vrx_path = os.path.join(self.data_path, row['csv_vrx_path'].lstrip('./'))
        
        vtx = pd.read_csv(vtx_path, skiprows=1) if vtx_path else None
        vrx = pd.read_csv(vrx_path, skiprows=1)
        
        return vtx, vrx
    
    def plot_comparative_analysis(self, row, start_time_s = 0, end_time_s=0.2):
        vtx, vrx = self.load_data_pair(row)
        vtx = self.normalize_signal(vtx)
        vrx = self.normalize_signal(vrx, window_size=7_000)
        
        mask_time = (vrx['Second'] >= start_time_s) & (vrx['Second'] <= end_time_s)
        
        fig = plt.figure(figsize=(18, 6))
        gs = fig.add_gridspec(1, 2, width_ratios=[3, 2])
        
        ax1 = fig.add_subplot(gs[0, 0])
        
        ax1.plot(vrx[mask_time]['Second'] * 10e3, vrx[mask_time]['Value'], 
                 label='VRX (Received)', color='tab:blue', alpha=0.4, lw=1)
        
        if vtx is not None:
            ax1.plot(vtx[mask_time]['Second'] * 10e3, vtx[mask_time]['Value'], 
                     label='VTX (Original)', color='tab:green', alpha=0.4, lw=1)
        
        ax1.set_title(f"Signal Comparison: Sample {row['idx']} ({row['freq_ghz']} GHz)")
        ax1.set_xlabel("Time (ms)")
        ax1.set_ylabel("Normalized Amplitude")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2 = fig.add_subplot(gs[0, 1])
        img_path = os.path.join(self.data_path, row['img_path'].lstrip('./'))
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            ax2.imshow(img)
            ax2.set_title("Visual Confirmation")
            ax2.axis('off')
        else:
            ax2.text(0.5, 0.5, "Image not found", ha='center', va='center')
            
        plt.tight_layout()
        plt.show()

    @staticmethod
    def normalize_signal(df, window_size=1_000_000):
        if df is None: return None
        data = df['Value'].values
        normalized = np.zeros_like(data)
    
        for i in range(0, len(data), window_size):
            end = min(i + window_size, len(data))
            window = data[i:end]

            normalized[i:end] = (window - np.min(window)) / (np.max(window) - np.min(window))

        new_df = df.copy()
        new_df['Value'] = normalized
        return new_df

    @staticmethod
    def calculate_snr(vtx_df, vrx_df):
        if vtx_df is None or vrx_df is None:
            return None

        s_ideal = vtx_df['Value'].values
        s_noisy = vrx_df['Value'].values

        noise = s_noisy - s_ideal
        
        p_signal = np.var(s_ideal)
        p_noise = np.var(noise)

        if p_noise == 0:
            return float('inf')

        snr_db = 10 * np.log10(p_signal / p_noise)
        return snr_db


loader = DroneDataLoader(data_path='..')

Registry successfully loaded. Available records: 104


# Methods
---
## Autocorrelation

In [ ]:
def compute_acf(signal_df):
    signal_values = signal_df['Value'].values
    signal_values = signal_values - np.mean(signal_values)

    acf_vals = signal.correlate(signal_values, signal_values, mode='full', method='auto')
    acf_vals = acf_vals[acf_vals.size // 2:]
    if acf_vals[0] != 0:
        acf_vals /= acf_vals[0]
    
    lags = np.arange(len(acf_vals)) * (signal_df['Second'].iloc[1] - signal_df['Second'].iloc[0])
    
    return lags, acf_vals

def autocorr_sample(vrx_sample, sync_window_us = range(60, 68), bg_window_us = list(range(28, 56)) + list(range(72, 100))):
    lags_t, acf_vals = compute_acf(vrx_sample)
    
    if acf_vals is None or len(acf_vals) == 0:
        return 0.0
        
    lags_us = lags_t * 1e6
    
    sync_vals = np.asarray(list(sync_window_us), dtype=float)
    bg_vals = np.asarray(list(bg_window_us), dtype=float)

    def _closest_mask(values):
        idx = np.array([np.argmin(np.abs(lags_us - v)) for v in values], dtype=int)
        mask = np.zeros(len(acf_vals), dtype=bool)
        mask[idx] = True
        return mask

    mask_sync = _closest_mask(sync_vals)
    mask_bg = _closest_mask(bg_vals)
    
    if not any(mask_sync) or not any(mask_bg):
        return 0.0
        
    max_sync = np.max(acf_vals[mask_sync])
    mean_sync = np.mean(acf_vals[mask_sync])
    
    avg_bg = np.mean(np.abs(acf_vals[mask_bg]))
    
    epsilon = 1e-6
    return (max_sync / (avg_bg + epsilon) > 3) | (mean_sync / (avg_bg + epsilon) > 1.65)

---
## FFT